# Week 1 Comparison Notebook

This notebook runs all four Week 1 methods, Benford, Isolation Forest, the
autoencoder, and the two supervised models, each through its own module, all
against the same frozen split. It saves the combined table to
`data/generated/model_comparison.csv`.


In [1]:
import pandas as pd

from finlint.anomaly.autoencoder import run_autoencoder
from finlint.anomaly.benford import run_benford
from finlint.anomaly.isolation_forest import run_isolation_forest
from finlint.anomaly.supervised import run_supervised


In [2]:
from pathlib import Path

# This makes the notebook work whether you open it from the repo root
# or from inside the notebooks folder.
if (Path.cwd() / "data").exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    REPO_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not find the repo root")

OUT_DIR = REPO_ROOT / "data" / "generated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)


Repo root: C:\Projects\FinLint


In [3]:
# Each method's own module runs its full pipeline through the same frozen
# split, so this notebook only needs to call all four and line up the results.
benford_summary = run_benford()

isolation_forest_summary = run_isolation_forest()["summary"]

autoencoder_summary = run_autoencoder()["summary"]
autoencoder_summary = {k: v for k, v in autoencoder_summary.items() if k != "threshold"}

supervised_results = run_supervised()["results_table"]

rows = [
    {**benford_summary, "notes": "unsupervised, group level, no labels used"},
    {**isolation_forest_summary, "notes": "unsupervised, engineered features"},
    {**autoencoder_summary, "notes": "unsupervised, trained on non-fraud rows only"},
]
for _, row in supervised_results.iterrows():
    rows.append({**row.to_dict(), "notes": "supervised, class_weight balanced"})

results_table = pd.DataFrame(rows)
results_table.to_csv(OUT_DIR / "model_comparison.csv", index=False)
results_table


,method,precision,recall,f1,n_flagged,notes,average_precision,roc_auc
0,Benford (MAD by gl_account),0.0638,0.0880,0.0739,11481,"unsupervised, group level, no labels used",NaN,NaN
1,Isolation Forest,0.4112,0.4100,0.4106,8296,"unsupervised, engineered features",NaN,NaN
2,Autoencoder,0.4179,0.5263,0.4659,10476,"unsupervised, trained on non-fraud rows only",NaN,NaN
3,Logistic regression,0.4189,0.6227,0.5008,12366,"supervised, class_weight balanced",0.6458,0.8259
4,Gradient boosting,0.7514,0.6049,0.6702,6697,"supervised, class_weight balanced",0.6660,0.8173
